# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides guidance on loading and exploring the [FAIR² Croissant](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL, conforming to the [mlcommons/croissant](https://mlcommons.org/croissant/) standard.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print a summary from the metadata
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values for informed data handling. This uses only metadata.

List all record sets and their fields, showing how to reference them by their `@id`.

In [ ]:
# List all record sets and fields (by @id) in the dataset
print("Available record sets and their fields:")
for rs in dataset.metadata.record_sets:
    print(f"RecordSet @id: {rs.id}")
    if hasattr(rs, 'name') and rs.name:
        print(f"  Name: {rs.name}")
    if hasattr(rs, 'description') and rs.description:
        print(f"  Description: {rs.description}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for field in rs.fields:
            print(f"    Field @id: {field.id} | Name: {getattr(field, 'name', '')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above for referencing.

Below, we select the first `recordSet` for demonstration, but you can adjust to your needs.

In [ ]:
# Extract data from all available record sets
record_sets = [rs.id for rs in dataset.metadata.record_sets]
record_set_names = {rs.id: getattr(rs, 'name', rs.id) for rs in dataset.metadata.record_sets}
dataframes = {}

for record_set_id in record_sets:
    print(f"Extracting records for RecordSet @id: {record_set_id} ({record_set_names[record_set_id]})")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  First 3 records:\n{df.head(3)}\n")
# For demonstration, select the first available record set (feel free to select another)
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id:
    print(f"Working record set: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records by value, normalizing a numeric field, removing outliers, and grouping, where appropriate.

In [ ]:
from pandas.api.types import is_numeric_dtype
# Identify numeric columns for potential EDA on the main record set
if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Pick the first numeric field that is not all null
    numeric_fields = [col for col in df.columns if is_numeric_dtype(df[col]) and df[col].notnull().sum() > 0]
    print(f"Numeric fields found: {numeric_fields}")
    if numeric_fields:
        numeric_field = numeric_fields[0]  # Choose first numeric field
        print(f"Proceeding with numeric field: {numeric_field}")
        # Choose a threshold for demonstration (median or 1st quartile value)
        threshold = df[numeric_field].quantile(0.5)
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold} (median): {len(filtered_df)} records")
        # Normalization (z-score)
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # If a non-numeric field exists, demonstrate grouping
        group_candidates = [col for col in df.columns if col != numeric_field and df[col].nunique() > 1]
        if group_candidates:
            group_field = group_candidates[0]
            # Group by the selected field and show mean of the numeric field
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable non-numeric field found for grouping.")
    else:
        print("No numeric fields found for EDA in this record set.")
else:
    print("No record set found for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and any relationships between fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and 'numeric_field' in locals():
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
    # If grouped_df is defined, plot group means
    if 'grouped_df' in locals():
        plt.figure(figsize=(8, 4))
        sns.barplot(
            data=grouped_df, x=grouped_df.columns[0], y=grouped_df.columns[1]
        )
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR² Croissant dataset using `mlcroissant`, including querying metadata, overviewing available data structures, extracting record data, performing basic filtering and grouping for analysis, and visualizing results.

For a full analysis, consider referencing the schema details and exploring other record sets as needed. All references to records, fields, and columns above use their stable `@id` identifiers.